# Train the tennis-ball detector

This notebook fine-tunes YOLO11l on version 6 of the [Tennis Ball Detection dataset](https://universe.roboflow.com/viren-dhanwani/tennis-ball-detection/dataset/6), published under CC BY 4.0. It follows the tutorial training workflow used by this project. Training 100 epochs can require substantial time and a GPU.

## 1. Install training dependencies

Ultralytics is pinned to the version recorded in the checkpoint used by this project.

In [ ]:
%pip install -q "ultralytics==8.3.179" roboflow

## 2. Download and validate the dataset

Set `ROBOFLOW_API_KEY` in the environment that starts Jupyter. Never paste the private key into this notebook or commit it to Git.

In [ ]:
import os
from pathlib import Path

from roboflow import Roboflow

api_key = os.environ.get("ROBOFLOW_API_KEY")
if not api_key:
    raise RuntimeError(
        "ROBOFLOW_API_KEY is not set. Add it to the Jupyter environment before continuing."
    )

roboflow = Roboflow(api_key=api_key)
project = roboflow.workspace("viren-dhanwani").project(
    "tennis-ball-detection"
)
dataset = project.version(6).download("yolov11")
dataset_root = Path(dataset.location)
data_yaml = dataset_root / "data.yaml"

required_paths = [
    data_yaml,
    dataset_root / "train" / "images",
    dataset_root / "valid" / "images",
    dataset_root / "test" / "images",
]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    formatted_paths = "\n".join(f"- {path}" for path in missing_paths)
    raise FileNotFoundError(f"Dataset download is incomplete:\n{formatted_paths}")

print(f"Dataset ready: {data_yaml}")

## 3. Fine-tune YOLO11l

The parameters below match the metadata stored in the local checkpoint used for the project benchmark. Exact weights can still vary across hardware and software environments.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11l.pt")
model.train(
    data=str(data_yaml),
    epochs=100,
    imgsz=640,
    batch=16,
    seed=0,
    deterministic=True,
)

best_checkpoint = Path(model.trainer.best)
print(f"Best checkpoint: {best_checkpoint}")

## 4. Use the checkpoint

Copy the generated `best.pt` file to `models/yolov11best.pt` in the repository before running the analysis pipeline. Model weights remain local and are ignored by Git.